In [1]:
import os
import re
from dotenv import load_dotenv, find_dotenv

env_path = find_dotenv(usecwd=True)          # searches cwd and upward
print(f"working directory   : {os.getcwd()}")
print(f"found .env at       : {env_path or 'NOT FOUND'}")

loaded = load_dotenv(env_path) if env_path else False
print(f"load_dotenv() said  : {loaded}")

if "GOOGLE_API_KEY" in os.environ:
    key = os.environ["GOOGLE_API_KEY"]
    print(f"key loaded          : {key[:6]}...{key[-4:]}   ({len(key)} characters)")
else:
    print("GOOGLE_API_KEY is NOT set — see troubleshooting below")

working directory   : /Users/wangyujia/Desktop/AI_SSR/yujia-first-repo
found .env at       : /Users/wangyujia/Desktop/AI_SSR/yujia-first-repo/.env
load_dotenv() said  : True
key loaded          : AQ.Ab8...A9fA   (53 characters)


In [2]:
from google import genai
from google.genai import types
client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

# 1. Choose an LLM, test with one call

In [13]:
MODEL = "gemini-3.7-flash"

r = client.models.generate_content(
    model=MODEL,
    contents="Reply with exactly one word: hello",
    config=types.GenerateContentConfig(
        temperature=0,
        max_output_tokens=5,
        automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
    ),
)
print(r.text)
print("tokens — in:", r.usage_metadata.prompt_token_count,
      "out:", r.usage_metadata.candidates_token_count)

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

# 2. Prepare Your Data

In [16]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split

#URL = ("https://raw.githubusercontent.com/yjin-chae/"
       #"LLMs-for-text-classification/main/data/original/semeval_2016.csv")
df = pd.read_csv("/Users/wangyujia/Desktop/AI_SSR/semeval_2016.csv", encoding="latin-1")

#df = pd.read_csv(URL, encoding="latin-1")           # the file has non-UTF-8 bytes
df = df[["Tweet", "Target", "Stance"]].reset_index(drop=True)

print(df.shape)
print("\nTarget x Stance:")
print(pd.crosstab(df["Target"], df["Stance"], margins=True))

(1691, 3)

Target x Stance:
Stance           AGAINST  FAVOR  NONE   All
Target                                     
Donald Trump         299    148   260   707
Hillary Clinton      565    163   256   984
All                  864    311   516  1691


In [17]:
strat = df["Target"] + "|" + df["Stance"]

train_df, test_df = train_test_split(df, test_size=0.30, random_state=42, stratify=strat)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

# 200-tweet held-out lab set. Everything below is scored on this.
lab_df = test_df.sample(n=200, random_state=42).reset_index(drop=True)

print(f"train {len(train_df)}   test {len(test_df)}   lab set {len(lab_df)}")
print("\nlab set Target x Stance:")
print(pd.crosstab(lab_df["Target"], lab_df["Stance"]))

train 1183   test 508   lab set 200

lab set Target x Stance:
Stance           AGAINST  FAVOR  NONE
Target                               
Donald Trump          42     20    31
Hillary Clinton       59     14    34


# 3. Prepare Your Prompt
## 3.1 Replicate the zero-shot prompt in the paper

In [18]:
LABELS = ["AGAINST", "FAVOR", "NONE"]

# The paper's exact zero-shot prompt, copied verbatim from prompt3 in the
# replication package. Note the trailing "\n\n" — the tweet gets appended after.
PROMPT = (
    "These statement contains a TARGET and a STANCE. The target is a politician "
    "and the stance represents the attitude expressed about them. The target "
    "options are Trump or Clinton and stance options are Favor, Against or None. "
    "Provide the answer in the following format: {TARGET, STANCE}\n\n"
)

In [19]:
tweet = "Some example tweet #SemST"
cleaned = tweet.replace("#SemST", "").strip()          # → "Some example tweet"

## 3.2 Check the actual prompt

In [20]:
row = lab_df.iloc[0]
tweet = row["Tweet"].replace("#SemST", "").strip()     # clean it, same as the paper does

print(PROMPT + tweet)

These statement contains a TARGET and a STANCE. The target is a politician and the stance represents the attitude expressed about them. The target options are Trump or Clinton and stance options are Favor, Against or None. Provide the answer in the following format: {TARGET, STANCE}

@3_Card_Monty : I believe the title would be "First Lord". Let THAT sink in!! #PJNET


## 3.3 Test call the prompt & annotation

In [38]:
MODEL = "gemini-3.1-flash-lite"

for k in range(5):
    row = lab_df.iloc[k]
    tweet = row["Tweet"].replace("#SemST", "").strip()
    r = client.models.generate_content(
        model=MODEL,
        contents=PROMPT + tweet,
        config=types.GenerateContentConfig(
            temperature=0,
            max_output_tokens=200,
            automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
        ),
    )
    reply = r.text
    print(f"[{k}] true=({row['Target']:15s}, {row['Stance']:8s})   "
          f"reply={reply.strip()[:80]!r}   ")

[0] true=(Hillary Clinton, NONE    )   reply='{None, None}'   
[1] true=(Hillary Clinton, NONE    )   reply='{Clinton, Against}'   
[2] true=(Donald Trump   , AGAINST )   reply='{Trump, Against}'   
[3] true=(Donald Trump   , NONE    )   reply='{None, None}'   
[4] true=(Hillary Clinton, AGAINST )   reply='{Clinton, Against}'   


# 4. Send API Requests for LLM Annotations

In [39]:
import time
def annotate_corpus(tweets, prompt=PROMPT, temperature=0):
    """Send every tweet to the model, one at a time. Returns a list of raw replies."""
    replies = []
    for tw in tweets:
        tweet = tw.replace("#SemST", "").strip()
        r = client.models.generate_content(
            model=MODEL,
            contents=prompt + tweet,
            config=types.GenerateContentConfig(
                temperature=temperature,
                max_output_tokens=200,
                automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
            ),
        )
        replies.append(r.text)
        if len(replies) % 10 == 0:
            print(f"  {len(replies)}/{len(tweets)} done")
            time.sleep(60)
    return replies

raw_zs = annotate_corpus(lab_df["Tweet"].tolist())
print(f"\n{len(raw_zs)} replies received. First one:\n{raw_zs[0]}")

  10/200 done
  20/200 done
  30/200 done
  40/200 done
  50/200 done
  60/200 done
  70/200 done
  80/200 done
  90/200 done
  100/200 done
  110/200 done
  120/200 done
  130/200 done
  140/200 done
  150/200 done
  160/200 done
  170/200 done
  180/200 done
  190/200 done
  200/200 done

200 replies received. First one:
{None, None}


# 5. Obtain LLM Annotations & Convert Text Label to Data

In [40]:
TARG_MAP   = {"trump": "Donald Trump", "clinton": "Hillary Clinton"}
STANCE_MAP = {"favor": "FAVOR", "against": "AGAINST", "none": "NONE"}

def parse(reply: str):
    """Extract (target, stance) from a model reply. Returns (None, None) on failure."""
    m = re.search(r"\{\s*([A-Za-z']+)\s*,\s*([A-Za-z]+)\s*\}", reply)
    if not m:
        return None, None
    target = TARG_MAP.get(m.group(1).lower().replace("'", ""))
    stance = STANCE_MAP.get(m.group(2).lower())
    return target, stance

parsed = [parse(r) for r in raw_zs]
n_bad_stance = sum(1 for _, s in parsed if s is None)
n_bad_target = sum(1 for t, _ in parsed if t is None)
print(f"unparseable stance : {n_bad_stance} / {len(parsed)}")
print(f"unparseable target : {n_bad_target} / {len(parsed)}")

unparseable stance : 0 / 200
unparseable target : 63 / 200


In [41]:
# Predictions -- default failures to NONE / the majority target (Clinton)
y_pred  = np.array([s if s in LABELS else "NONE" for _, s in parsed])
t_pred  = np.array([t if t is not None else "Hillary Clinton" for t, _ in parsed])
y_true  = lab_df["Stance"].to_numpy()
targets = lab_df["Target"].to_numpy()

lab_df_out = lab_df.copy()
lab_df_out["stance_pred"] = y_pred
lab_df_out["target_pred"] = t_pred
lab_df_out.to_csv("lab3_annotations.csv", index=False)
print(lab_df_out.head())

                                               Tweet           Target  \
0  @3_Card_Monty : I believe the title would be "...  Hillary Clinton   
1  @FrankCraig: @skzdalimit if you're the best yo...  Hillary Clinton   
2  What does Ivana, NBC, Univision, Gemini's, and...     Donald Trump   
3  If the Obama cabinet was on the apprentice who...     Donald Trump   
4  'Hillary-speak' campaign rhetoric is going to ...  Hillary Clinton   

    Stance stance_pred      target_pred  
0     NONE        NONE  Hillary Clinton  
1     NONE     AGAINST  Hillary Clinton  
2  AGAINST     AGAINST     Donald Trump  
3     NONE        NONE  Hillary Clinton  
4  AGAINST     AGAINST  Hillary Clinton  


# 6. Evaluate
## 6.1 Evaluate performance on predicting TARGET

In [42]:
target_acc = (t_pred == targets).mean()

# Overall accuracy
print(f"target inferred correctly: {(t_pred == targets).sum()}/{len(targets)}  "
      f"({target_acc:.1%})")

# The Confusion Matrix
print("\nTarget confusion matrix:")
print(pd.crosstab(pd.Series(targets, name="true"),
                  pd.Series(t_pred, name="predicted"), margins=True))

target inferred correctly: 154/200  (77.0%)

Target confusion matrix:
predicted        Donald Trump  Hillary Clinton  All
true                                               
Donald Trump               52               41   93
Hillary Clinton             5              102  107
All                        57              143  200


## 6.2 Evaluate performance on predicting STANCE

In [43]:
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, precision_recall_fscore_support,
                             cohen_kappa_score)

print(f"n = {len(y_true)}\n")
print(f"accuracy       : {accuracy_score(y_true, y_pred):.3f}")
print(classification_report(y_true, y_pred, labels=LABELS, digits=3))

n = 200

accuracy       : 0.760
              precision    recall  f1-score   support

     AGAINST      0.885     0.683     0.771       101
       FAVOR      0.667     0.941     0.780        34
        NONE      0.689     0.785     0.734        65

    accuracy                          0.760       200
   macro avg      0.747     0.803     0.762       200
weighted avg      0.784     0.760     0.761       200



In [44]:
cm = pd.DataFrame(confusion_matrix(y_true, y_pred, labels=LABELS),
                  index=[f"true_{l}" for l in LABELS],
                  columns=[f"pred_{l}" for l in LABELS])
print(cm)

              pred_AGAINST  pred_FAVOR  pred_NONE
true_AGAINST            69          10         22
true_FAVOR               1          32          1
true_NONE                8           6         51


## 6.3 STANCE Performance by TARGET

In [45]:
for t in ["Donald Trump", "Hillary Clinton"]:
    m = targets == t
    p, r, f, _ = precision_recall_fscore_support(y_true[m], y_pred[m], labels=LABELS,
                                                 average="macro", zero_division=0)
    acc = accuracy_score(y_true[m], y_pred[m])
    print(f"{t:18s}  n={m.sum():3d}   acc {acc:.3f}   macro Precision {p:.3f}   "
          f"macro Recall {r:.3f}   macro F1 {f:.3f}")

Donald Trump        n= 93   acc 0.710   macro Precision 0.718   macro Recall 0.757   macro F1 0.723
Hillary Clinton     n=107   acc 0.804   macro Precision 0.760   macro Recall 0.834   macro F1 0.782


## 6.4 Joint TARGET + STANCE performance

In [46]:
# Treat (target, stance) as a single composite class label. sklearn wants a plain
# scalar label per sample, so we join the two fields into one string like
# "Donald Trump|AGAINST" rather than using a Python tuple.
joint_true = np.array([f"{t}|{s}" for t, s in zip(targets, y_true)])
joint_pred = np.array([f"{t}|{s}" for t, s in zip(t_pred,  y_pred)])

JOINT_LABELS = [f"{t}|{s}"
                for t in ["Donald Trump", "Hillary Clinton"]
                for s in LABELS]

joint_acc = (joint_true == joint_pred).mean()
p, r, f, _ = precision_recall_fscore_support(
    joint_true, joint_pred, labels=JOINT_LABELS,
    average="macro", zero_division=0)

print(f"joint accuracy  : {joint_acc:.3f}   "
      f"(both target AND stance correct on {int((joint_true == joint_pred).sum())} / "
      f"{len(joint_true)} tweets)")
print(f"joint macro F1  : {f:.3f}   (macro-averaged across all 6 (target, stance) classes)")

joint accuracy  : 0.615   (both target AND stance correct on 123 / 200 tweets)
joint macro F1  : 0.589   (macro-averaged across all 6 (target, stance) classes)


In [47]:
# Where does the joint metric drop below §6.2's stance-only F1?
stance_correct = (y_pred == y_true)
target_correct = (t_pred == targets)
both           = stance_correct & target_correct
print(f"of {len(y_true)} tweets:")
print(f"  stance correct only : {(stance_correct & ~target_correct).sum():3d}")
print(f"  target correct only : {(~stance_correct & target_correct).sum():3d}")
print(f"  both correct        : {both.sum():3d}")
print(f"  both wrong          : {(~stance_correct & ~target_correct).sum():3d}")

of 200 tweets:
  stance correct only :  29
  target correct only :  31
  both correct        : 123
  both wrong          :  17


# 8. Optional: Annotating an Image

In [48]:
# The image lives in the course repo under weeks/week03/images/. 
# You might need to change the path depending on your file structure.
# To try a different image, drop any .jpg into that folder and update IMG_PATH.
IMG_PATH = "/Users/wangyujia/Desktop/AI_SSR/yujia-first-repo/Suburban_town_houses.jpg"

FEATURES = ["single-family houses", "apartment buildings", "highway", "fencing",
            "sidewalk", "dense vegetation", "parking lot", "railroad track"]

with open(IMG_PATH, "rb") as f:
    image_bytes = f.read()

image_part = types.Part.from_bytes(data=image_bytes, mime_type="image/jpeg")

r = client.models.generate_content(
    model=MODEL,                                     # Gemini Flash handles images natively
    contents=[
        "Which of the listed features appear in this image?",
        image_part,
    ],
    config=types.GenerateContentConfig(
        system_instruction=(
            "You label built-environment features in images for a research project. "
            f"Use ONLY these labels: {FEATURES}. "
            'Return JSON: {"features": [...]} and nothing else.'
        ),
        temperature=0,
        max_output_tokens=500,
        automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
    ),
)
print(r.text)

{"features": ["single-family houses", "fencing", "sidewalk", "dense vegetation"]}
